In [9]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

In [10]:
# Load models and preprocessing objects

model_b = tf.keras.models.load_model("lstm_review_classifier")  # Model B
with open("tokenizer.pkl", "rb") as f:
    tokenizer_b = pickle.load(f)
label_encoder_b = joblib.load("label_encoder.pkl")

model_a = tf.keras.models.load_model("lstm_balanced_review_model1")  # Model A
tokenizer_a = joblib.load("tokenizer_lstm_balanced.pkl")
label_encoder_a = joblib.load("label_encoder_lstm_balanced.pkl")

In [11]:
# Load & split balanced dataset

balanced_df = pd.read_csv("addbalanced_data.csv")
train_bal, test_bal = train_test_split(
    balanced_df, test_size=0.2, random_state=42, stratify=balanced_df["Rating"]
)
balanced_texts_test = test_bal["Review"].astype(str).tolist()
balanced_labels_test = label_encoder_a.transform(test_bal["Rating"])

In [12]:
# Load & split imbalanced dataset

imbalanced_df = pd.read_csv("addimbalanced_dataset.csv")
train_imb, test_imb = train_test_split(
    imbalanced_df, test_size=0.2, random_state=42, stratify=imbalanced_df["Rating"]
)
imbalanced_texts_test = test_imb["Review"].astype(str).tolist()
imbalanced_labels_test = label_encoder_b.transform(test_imb["Rating"])

In [13]:
# Evaluation Function

def evaluate_model(model, X_test, y_test, label_encoder, model_name, test_set_name):
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    target_names = [str(cls) for cls in label_encoder.classes_]

    print(f"\n=== {model_name} on {test_set_name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=target_names))

In [14]:
#  Model A on Imbalanced test set 
imbalanced_sequences_for_A = tokenizer_a.texts_to_sequences(imbalanced_texts_test)
imbalanced_padded_for_A = tf.keras.preprocessing.sequence.pad_sequences(
    imbalanced_sequences_for_A, maxlen=128, padding='post', truncating='post'
)
evaluate_model(model_a, imbalanced_padded_for_A, imbalanced_labels_test,
               label_encoder_a, "Model A (Balanced)", "Imbalanced Test Set")


#  Model B on Balanced test set 
balanced_sequences_for_B = tokenizer_b.texts_to_sequences(balanced_texts_test)
balanced_padded_for_B = tf.keras.preprocessing.sequence.pad_sequences(
    balanced_sequences_for_B, maxlen=128, padding='post', truncating='post'
)
evaluate_model(model_b, balanced_padded_for_B, balanced_labels_test,
               label_encoder_b, "Model B (Imbalanced)", "Balanced Test Set")



=== Model A (Balanced) on Imbalanced Test Set ===
Accuracy: 0.5009
Precision: 0.5115
Recall: 0.5009
F1-score: 0.4939

Confusion Matrix:
 [[2986 1628  257   46   83]
 [2074 3818 1158  266  184]
 [1125 3695 4261 2443  976]
 [ 350 1095 2977 6316 4262]
 [ 174  188  400 1573 7665]]

Classification Report:
               precision    recall  f1-score   support

         1.0       0.45      0.60      0.51      5000
         2.0       0.37      0.51      0.43      7500
         3.0       0.47      0.34      0.40     12500
         4.0       0.59      0.42      0.49     15000
         5.0       0.58      0.77      0.66     10000

    accuracy                           0.50     50000
   macro avg       0.49      0.53      0.50     50000
weighted avg       0.51      0.50      0.49     50000


=== Model B (Imbalanced) on Balanced Test Set ===
Accuracy: 0.5227
Precision: 0.5093
Recall: 0.5227
F1-score: 0.5125

Confusion Matrix:
 [[6605 2203  811  179  202]
 [3515 3537 2077  599  272]
 [1127 2026 3